# Projet MTH2329 — Méta-modèle par espace latent : de la réduction de dimension à la probabilité de dépassement de seuil

**UE MTH2329 — Data Science et IA** · Master IA-BD · Université de Lomé · École Polytechnique de Lomé

## Contexte

On s'intéresse à la construction d'un **méta-modèle** (modèle de substitution) pour un simulateur coûteux

$$F : (\mu, \sigma^2) \in \mathbb{R}^2 \;\longrightarrow\; \phi(\,\cdot\,; \mu, \sigma^2) \in \mathbb{R}^{M},$$

où $\phi(x; \mu, \sigma^2)$ est la densité gaussienne de moyenne $\mu$ et de variance $\sigma^2$, discrétisée sur $M = 800$ points de $[-10, 10]$. Les paramètres d'entrée sont aléatoires : $\mu \sim \mathcal{U}[-4, 4]$, $\sigma^2 \sim \Gamma(k=2, \theta=2)$.

L'objectif final est de produire, **à faible coût**, la **probabilité de dépassement d'un seuil** $\tau$ en chaque position $x$ :

$$E(x,\tau) = \mathbb{P}\big(\phi(x; \mu, \sigma^2) \geq \tau\big),$$

quantité qu'une simulation Monte Carlo « brute » du modèle physique rendrait inabordable.

**Pipeline étudié** (3 étapes) :

$$(\mu, \sigma^2) \;\xrightarrow{\text{GPR}}\; \mathbf{z} \in \mathbb{R}^d \;\xrightarrow{\text{décodeur}}\; \hat\phi(x; \mu, \sigma^2) \;\xrightarrow{L \text{ tirages}}\; \hat p(x) \approx E(x).$$

## Objectifs pédagogiques

1. **Réduction de dimension** : PCA (scikit-learn), autoencodeur dense (PyTorch *ou* tensorflow, au choix), comparaison avec d'autres méthodes.
2. **Choix de la dimension latente** $d$ (étude comparative).
3. **Régression par processus gaussiens (GPR)** dans l'espace latent : choix du noyau, optimisation des hyperparamètres.
4. **Probabilité de dépassement** par Monte Carlo à travers le méta-modèle, comparaison à une référence exacte obtenue par intégration numérique.

## Consignes générales (à lire avant de commencer)

- **Structure du code** : la plus grande partie du code (génération des données, classes de réduction, GPR, dépassement) doit être écrite dans des **fichiers Python (`.py`)** regroupés dans le dossier `src/`, puis **importée** dans le notebook. Le notebook ne doit contenir que l'orchestration (appels de fonctions, boucles d'expériences, figures) et vos réponses. Des squelettes vous sont fournis dans `src/` : complétez les fonctions marquées `# TODO`.
- **Librairies** : `numpy`, `scipy`, `scikit-learn` (PCA et GPR), `matplotlib`. Pour l'autoencodeur, **vous êtes libres** d'utiliser **PyTorch ou TensorFlow/Keras** — les deux approches sont valorisées à parts égales. Précisez votre choix en début de partie 1.2.
- **Reproductibilité** : fixez les graines (`np.random.seed(42)`, `torch.manual_seed(42)` ou `tf.random.set_seed(42)`) ; vos figures doivent être titrées, légendées, avec axes nommés.
- **Réponses** : chaque question `Q…` doit être traitée dans la **cellule Markdown** prévue à cet effet (bloc « Réponse »). Une réponse = une explication concise argumentée (2–6 phrases), éventuellement appuyée par un chiffre ou une figure.
- **Durée indicative** : 3 séances de 2 h. Les parties marquées *(bonus)* sont hors barème principal mais valorisées.

## Rendu attendu

Un dossier `Nom1_Nom2/` contenant :
- **le notebook complété** (figures + réponses) et le dossier `src/` avec vos modules `.py` documentés, à déposer **sur Moodle** (un seul fichier `.ipynb` + `src/` en archive) ;
- aucun rapport n'est demandé : l'ensemble des résultats et des discussions doit figurer dans le notebook.

**Barème indicatif** : Partie 1 (6 pts) · Partie 2 (6 pts) · Partie 3 (5 pts) · qualité du code et des réponses (3 pts).

## Partie 0 — Génération et exploration des données

Les cellules de cette partie sont **fournies fonctionnelles** : exécutez-les, puis répondez aux questions.

> Remarque : la génération des données est codée dans `src/data.py`. Vérifiez son contenu — c'est le premier module que vous devez comprendre (et respecter) pour la suite.

In [ ]:
#Pour recharger automatiquement les modules modifiés dans un notebook Jupyter
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('./src')

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# dimensions du probleme
M = 800                    # nombre de points de discretisation
N = 2000                   # nombre de courbes
mu_min, mu_max = -4., 4.   # mu ~ U[mu_min, mu_max]
k_shape, theta_scale = 2., 2.   # sigma^2 ~ Gamma(k, theta)

from src.data import gaussian_pdf, make_dataset

x_grid, y, X = make_dataset(N=N, M=M, mu_min=mu_min, mu_max=mu_max,
                            k_shape=k_shape, theta_scale=theta_scale, seed=42)

# decoupage train / test (80 / 20)
from sklearn.model_selection import train_test_split
y_train, y_test, X_train, X_test = train_test_split(y, X, test_size=0.2, random_state=42)

print(f"x_grid : {x_grid.shape}  |  y : {y.shape}  |  X : {X.shape}")
print(f"train : {y_train.shape}  |  test : {y_test.shape}")

In [ ]:
# visualisation d'un echantillon de 100 courbes
from src.visualization import add_caption                                  
fig, ax = plt.subplots(figsize=(12, 4))
for i in np.random.choice(len(y), 100, replace=False):
    ax.plot(x_grid, y[i], lw=0.7, alpha=0.5)
ax.set_xlabel(r'$x$'); ax.set_ylabel(r'$\phi(x; \mu, \sigma^2)$')
ax.set_title('Echantillon de 100 courbes gaussiennes')
add_caption(fig, 'Échantillon de 100 courbes du jeu de données.') 
plt.show()

In [ ]:
from src.visualization import plot_curve_points, add_caption

i0 = 580   # numéro de la courbe déssinée
fig = plot_curve_points(x_grid, y_train[i0],
                        mu=X_train[i0, 0], var=X_train[i0, 1],
                        name=f'courbe n°{i0} du train')
add_caption(fig, f'Une courbe du train (n°{i0}) dessinée avec ses 800 points de discrétisation.')
plt.show()

### Questions — Partie 0

**Q0.1.** Quelles sont les dimensions respectives de l'espace d'entrée $\mathcal{X}$ et de l'espace de sortie $\mathcal{Y}$ ? Pourquoi une régression directe de $\mathcal{X}$ vers $\mathcal{Y}$ (par un GPR classique, sans espace latent) serait-elle problématique ?

**Q0.2.** Justifiez l'intérêt du découpage train/test. Quelle fraction des données est utilisée pour évaluer le méta-modèle final ?

**Réponse Q0.1 :**
- L'espace d'entrée $\mathcal{X}$ a la forme (2000, 2); chaque ligne est un couple $(\mu, \sigma^2)$ donc de **dimension 2**, avec $\mathcal{X} \subset \mathbb{R}^2$ et $(\mu, \sigma^2) \in [-4, 4] \times \mathbb{R}_+^*$.L'espace de sortie $\mathcal{Y}$ a la forme (2000, 800); chaque ligne est une courbe de 800 points. On peut d'ailleurs voire sur la figure ci-dessus comment est représentée une courbe par ces points. Il est donc de **dimension 800**, avec $\mathcal{Y} \subset \mathbb{R}^{800}$.

- Une régression directe de $\mathcal{X}$ vers $\mathcal{Y}$ par GPR classique serait problématique parce que 800 sorties, c'est lourd, et que ces valeurs sont très liées entre elles. Comme les courbes ne dépendent que de 2 paramètres $(\mu, \sigma^2)$ , il est plus efficace de les compresser d'abord dans un espace latent de petite dimension d, puis de faire la régression vers ce code.


**Réponse Q0.2 :** 
- Le découpage **train/test** mesure **la capacité du méta-modèle à généraliser** à des paramètres $(\mu, \sigma^2)$ qu'il n'a jamais vus.L'apprentissage se fait donc uniquement sur le train et **20 % des données**, soit 400 courbes, servent à évaluer le méta-modèle final.Le tirage aléatoire **(random_state=42)** garantit que les deux ensembles suivent la même loi d'entrée.

## Partie 1 — Réduction de dimension des courbes

Chaque courbe vit dans $\mathbb{R}^{800}$ mais est entièrement déterminée par **2 paramètres**. L'hypothèse de travail est donc qu'elle se concentre près d'une variété de dimension faible. Nous allons comparer plusieurs réducteurs $R : \mathbb{R}^{800} \to \mathbb{R}^d$ (avec reconstruction $R^{-1}$) pour plusieurs valeurs de $d$.

**Critère de comparaison principal** : l'erreur quadratique moyenne de reconstruction sur le jeu de test,

$$\mathrm{EQM}(d) = \frac{1}{n_{\text{test}}}\sum_{i} \big\| y_i - \hat y_i \big\|^2 / M, \qquad \hat y_i = R^{-1}(R(y_i)).$$

> Toutes les classes de réduction doivent proposer une **interface commune** (`fit(Y)`, `transform(Y)`, `inverse_transform(Z)`) afin que le reste du projet soit indépendant de la méthode choisie. Complétez le squelette `src/reduction.py`.

### 1.1 — PCA (scikit-learn)

La PCA est linéaire : c'est la référence à battre. 

Consignes :
1. Complétez la classe `PCAReducer` dans `src/reduction.py` (interface `fit` / `transform` / `inverse_transform`, paramètre `n_components`).
2. Pour $d \in \{1, 2, 3, 5, 10, 20, 50\}$, calculez l'EQM de reconstruction **sur le jeu de test**.
3. Représentez $\mathrm{EQM}(d)$ en échelle semi-log, et superposez la courbe du **ratio de variance expliquée cumulée**.
4. Affichez côte à côte, pour $d = 2$, une courbe test et sa reconstruction PCA.

```python
# --- snippet d'aide (PCA) ---
from sklearn.decomposition import PCA

pca = PCA(n_components=d)
pca.fit(y_train)                       # apprentissage (uniquement sur le train!)
z = pca.transform(y_test)              # projection
y_hat = pca.inverse_transform(z)       # reconstruction
print(pca.explained_variance_ratio_.cumsum())   # variance expliquee cumulee
```

In [ ]:
from src.reduction import PCAReducer, reconstruction_mse, cumulative_explained_variance
from src.visualization import mse_table, plot_mse_vs_d, plot_reconstructions, plot_pca_components, add_caption

# 1) EQM de reconstruction pour chaque d, sur le train et sur le test
d_values = [1, 2, 3, 5, 10, 20, 50]
pca_reducers, mse_pca_train, mse_pca_test = {}, [], []
for d in d_values:
    pca = PCAReducer(n_components=d).fit(y_train)     
    pca_reducers[d] = pca
    mse_pca_train.append(reconstruction_mse(pca, y_train))
    mse_pca_test.append(reconstruction_mse(pca, y_test))
    

# 2) Variance expliquée cumulée et nombre de composantes pour 99 %
cum_var = cumulative_explained_variance(y_train, n_max=60)
d_99 = int(np.searchsorted(cum_var, 0.99) + 1)
print(f"Composantes nécessaires pour 99 % de variance : {d_99}")

display(mse_table(d_values, mse_pca_train, mse_pca_test, cum_var=cum_var))

# 3) Courbe EQM(d) + variance cumulée
ax = plot_mse_vs_d({'PCA (test)': (d_values, mse_pca_test),
               'PCA (train)': (d_values, mse_pca_train)}, cum_var=cum_var, threshold=0.99)
add_caption(ax, 'EQM de reconstruction de la PCA (train et test) en fonction de d, '
                'et variance expliquée cumulée.')
plt.show()

# 4) Comparaison visuelle pour d = 2 (et d = 10) : une courbe étroite, une typique, une large
idx = [int(np.argmin(np.abs(X_test[:, 1] - v))) for v in (0.3, 3.0, 12.0)]
titles = [fr'test n°{i} : $\mu$ = {X_test[i, 0]:.2f}, $\sigma^2$ = {X_test[i, 1]:.2f}' for i in idx]
recons = {f'PCA d = {d}': pca_reducers[d].inverse_transform(pca_reducers[d].transform(y_test[idx]))
          for d in (2, 10)}
fig = plot_reconstructions(x_grid, y_test[idx], recons, titles=titles)
add_caption(fig, 'Trois courbes de test (étroite, typique, large) et leurs reconstructions '
                 'PCA pour d = 2 et d = 10.')
plt.show()

# 5) Complément : les « courbes de base » de la PCA
fig = plot_pca_components(x_grid, pca_reducers[5], n_components=5)
add_caption(fig, 'Courbe moyenne et 5 premières composantes principales de la PCA.')
plt.show()

**Q1.1.** Combien de composantes la PCA nécessite-t-elle pour expliquer 99 % de la variance ? Ce nombre est-il cohérent avec la dimension de l'espace des paramètres (2) ? Interprétez l'écart.

**Q1.2.** La PCA est-il un candidat satisfaisant pour le méta-modèle ? Quel est son défaut principal ici ?

**Réponse Q1.1 :**
- La PCA nécessite **10 composantes** pour expliquer 99 % de la variance. Avec $d = 2$, elle n'en explique que **77,77%**, alors que les courbes ne dépendent que de 2 paramètres. Ce n'est donc pas cohérent avec la dimension intrinsèque.

- Cet écart est dû au fait que la PCA cherche **le meilleur sous-espace linéaire**, alors que les courbes forment **une surface courbe de dimension 2** dans $\mathbb{R}^{800}$ : faire varier $\mu$ translate la cloche, ce qui n'est pas une opération linéaire. Pour suivre cette surface avec des combinaisons linéaires, la PCA **doit empiler des composantes de plus en plus oscillantes** (voir la figure des composantes). Elle nécessite donc beaucoup plus de dimensions **(≈ 10 au lieu de 2)** pour représenter correctement les courbes.


**Réponse Q1.2 :**
- La PCA est rapide, déterministe, et possède un décodeur exact et linéaire. C'est une bonne référence, mais **pas un candidat satisfaisant pour un espace latent compact**. **Son défaut principal est sa linéarité** : aux petites dimensions ($d = 2$ ou $3$), les reconstructions sont fausses et non physiques. On voit sur la **Figure 4** que les courbes reconstruites avec $d = 2$ n'ont pas les mêmes pics que les courbes originales (le pic de la courbe étroite passe de 0,73 à 0,27) et qu'elles descendent sous 0, ce qui est impossible pour une densité. Or ces erreurs sur la hauteur du pic faussent directement l'estimation de $\mathbb{P}(\phi \geq \tau)$.

Pour être précise, la PCA doit monter à $d \geq 10$–$20$. La GPR devra alors prédire 10 à 20 codes, dont les derniers correspondent aux composantes les plus oscillantes et seront probablement plus difficiles à apprendre.

### 1.2 — Autoencodeur dense (PyTorch *ou* tensorflow)

Un autoencodeur apprend une réduction **non linéaire** : encodeur $g_\phi : \mathbb{R}^{800} \to \mathbb{R}^d$, décodeur $f_\theta : \mathbb{R}^d \to \mathbb{R}^{800}$, entraînés conjointement par erreur de reconstruction (MSE).

**Indiquez ici votre choix de librairie** : ☐ PyTorch ☑ **tensorflow** (Keras)

Consignes :
1. Complétez la classe `AutoEncoderReducer` dans `src/reduction.py`. Architecture suggérée (à adapter) : `800 → 128 → 64 → d → 64 → 128 → 800`, activation SiLU/ReLU, sortie linéaire, loss MSE, optimiseur Adam ($10^{-3}$), early stopping sur la validation, `batch_size = 64`.
2. Entraînez pour $d \in \{2, 3, 5, 10, 20\}$ ; stockez à chaque fois l'EQM test de reconstruction.
3. Tracez $\mathrm{EQM}_{\text{AE}}(d)$ et superposez la courbe PCA de la question 1.1.
4. Pour $d = 2$ : visualisez l'espace latent (codes des courbes de train, colorés par $\mu$ puis par $\sigma^2$) et commentez l'organisation obtenue.

```python
# --- squelette PyTorch (si vous choisissez PyTorch) ---
import torch, torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, nbx=800, nbz=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nbx, 128), nn.SiLU(),
            nn.Linear(128, 64),  nn.SiLU(),
            nn.Linear(64, nbz),
        )
    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):   # symetrique de l'encodeur
    ...

# boucle d'entrainement : descente de gradient sur la MSE,
# jeu de validation = 20 % du train, early stopping.
```

In [ ]:
import time
from src.reduction import AutoEncoderReducer
from src.visualization import (compare_table, plot_mse_vs_d, plot_training_history,
                               plot_latent_space, plot_reconstructions, add_caption)

# 1) Entraînement de l'AE pour chaque d (+ PCA aux mêmes d pour comparer)
d_values_ae = [2, 3, 5, 10, 20]
ae_reducers, mse_ae_test, time_ae = {}, [], []
mse_pca_ae, time_pca = [], []
for d in d_values_ae:
    ae = AutoEncoderReducer(nbx=M, latent_dim=d, epochs=300, batch_size=64, seed=42).fit(y_train)
    ae_reducers[d] = ae
    mse_ae_test.append(reconstruction_mse(ae, y_test))
    time_ae.append(ae.fit_time_)
    t0 = time.perf_counter()
    pca = PCAReducer(n_components=d).fit(y_train)
    time_pca.append(time.perf_counter() - t0)
    mse_pca_ae.append(reconstruction_mse(pca, y_test))
    print(f"d = {d:2d} : {ae.n_epochs_} époques, {ae.fit_time_:.0f} s")

display(compare_table(d_values_ae, mse_pca_ae, mse_ae_test, time_pca=time_pca, time_ae=time_ae))

# 2) EQM_AE(d) superposée à la courbe PCA
ax = plot_mse_vs_d({'PCA (test)': (d_values, mse_pca_test),
                    'AE (test)': (d_values_ae, mse_ae_test)})
add_caption(ax, "EQM de reconstruction sur le test : autoencodeur comparé à la PCA.")
plt.show()

# 3) Courbes d'apprentissage (coût d'entraînement)
ax = plot_training_history({d: ae_reducers[d].history_ for d in d_values_ae})
add_caption(ax, "Courbes d'apprentissage des autoencodeurs pour chaque d.")
plt.show()

# 4) Espace latent pour d = 2, coloré par mu puis par sigma^2
fig = plot_latent_space(ae_reducers[2].transform(y_train), X_train,
                        title="Espace latent de l'autoencodeur (d = 2), courbes du train")
add_caption(fig, "Codes latents de l'autoencodeur pour d = 2, colorés par μ puis par σ².")
plt.show()

# 5) Reconstructions pour d = 2 : PCA et AE sur les mêmes courbes de test
idx = [int(np.argmin(np.abs(X_test[:, 1] - v))) for v in (0.3, 3.0, 12.0)]
titles = [fr'test n°{i} : $\mu$ = {X_test[i, 0]:.2f}, $\sigma^2$ = {X_test[i, 1]:.2f}' for i in idx]
recons = {'PCA d = 2': pca_reducers[2].inverse_transform(pca_reducers[2].transform(y_test[idx])),
          'AE d = 2': ae_reducers[2].inverse_transform(ae_reducers[2].transform(y_test[idx]))}
fig = plot_reconstructions(x_grid, y_test[idx], recons, titles=titles)
add_caption(fig, "Trois courbes de test et leurs reconstructions pour d = 2 : PCA et autoencodeur.")
plt.show()

**Q1.3.** Comparez quantitativement PCA et autoencodeur : pour quelles dimensions $d$ l'AE bat-il la PCA, et par quel facteur ? Que dire du compromis précision / coût d'entraînement ?

**Réponse Q1.3 :**

**Comparaison quantitative.** L'autoencodeur bat la PCA aux petites dimensions : son EQM test est $6{,}4$ fois plus faible à $d = 2$, $4{,}5$ fois à $d = 3$ et $2{,}4$ fois à $d = 5$. Les deux méthodes sont équivalentes à $d = 10$ (facteur $1{,}1$), et la PCA reprend l'avantage à $d = 20$ (EQM $2{,}2$ fois plus faible que l'AE).

L'erreur de l'AE dépend très peu de $d$ (de $1{,}7 \cdot 10^{-4}$ à $d = 2$ à $1{,}3 \cdot 10^{-4}$ à $d = 20$) : grâce à sa non-linéarité, il décrit déjà la surface des courbes avec 2 dimensions, la dimension intrinsèque du problème. Son erreur restante est limitée par la taille du réseau et par l'entraînement plutôt que par $d$ : les 5 entraînements ont atteint la limite de 300 époques sans déclencher l'early stopping, donc un entraînement plus long pourrait encore l'améliorer. Ainsi, **l'AE à $d = 2$ est aussi précis qu'une PCA à $d = 10$** ($1{,}66 \cdot 10^{-4}$ contre $1{,}46 \cdot 10^{-4}$).

**Compromis précision / coût.** L'AE demande 64 à 70 s d'entraînement, contre moins de 1 s pour la PCA (environ 70 à 120 fois plus). Mais ce coût n'est payé qu'une fois, hors ligne, et reste négligeable devant un seul appel au simulateur, supposé très couteux. Pour le méta-modèle, l'AE à petit $d$ est donc le meilleur compromis : la GPR n'aura que 2 codes à prédire, au lieu d'une dizaine avec la PCA pour une précision comparable.

**Choix de TensorFlow / Keras.** L'API de haut niveau (`compile` / `fit`) évite d'écrire la boucle d'entraînement. L'early stopping et la réduction du pas d'apprentissage sont fournis directement sous forme de *callbacks*. Le décodeur se définit comme un sous-modèle séparé, donc réutilisable seul en Parties 2 et 3. Enfin, toutes les graines se fixent en une ligne (`keras.utils.set_random_seed`).

### 1.3 — *(Bonus)* Autres méthodes de réduction

Pour aller plus loin, vous pouvez comparer avec :

- **Kernel PCA** (`sklearn.decomposition.KernelPCA`, noyau `rbf`) — non linéaire, mais le choix du noyau et de `gamma` est délicat ;
- **t-SNE / UMAP** — excellentes pour la visualisation, mais **pas de `inverse_transform`** : discutez de la conséquence pour un méta-modèle ;
- **Autoencodeur variationnel (VAE)** — espace latent probabiliste régularisé (module `src/` du démonstrateur d'enseignant, mentionné à titre indicatif).

```python
# --- snippet d'aide ---
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(n_components=d, kernel='rbf', gamma=0.05, fit_inverse_transform=True)
```

In [ ]:
import time
from sklearn.manifold import TSNE
from src.reduction import tune_kernel_pca
from src.visualization import (models_table, plot_mse_vs_d, plot_latent_space,
                               plot_reconstructions, add_caption)

# 1) Kernel PCA (noyau RBF) : gamma et alpha choisis sur une validation tirée du train
kpca_reducers, mse_kpca_test = {}, []
for d in d_values_ae:
    kpca, scores = tune_kernel_pca(y_train, n_components=d)
    kpca_reducers[d] = kpca
    mse_kpca_test.append(reconstruction_mse(kpca, y_test))
    print(f"d = {d:2d} : gamma = {kpca.gamma:g}, alpha = {kpca.alpha:g}, "
          f"EQM test = {mse_kpca_test[-1]:.3e}, ajustement {kpca.fit_time_:.2f} s")

# 2) EQM(d) des trois méthodes qui savent reconstruire
ax = plot_mse_vs_d({'PCA (test)': (d_values, mse_pca_test),
                    'AE (test)': (d_values_ae, mse_ae_test),
                    'Kernel PCA (test)': (d_values_ae, mse_kpca_test)})
add_caption(ax, "EQM de reconstruction sur le test : PCA, autoencodeur et Kernel PCA.")
plt.show()

# 3) Reconstructions pour d = 2 (mêmes courbes de test qu'en 1.1 et 1.2)
recons = {f'{nom} d = 2': r.inverse_transform(r.transform(y_test[idx]))
          for nom, r in [('PCA', pca_reducers[2]), ('Kernel PCA', kpca_reducers[2]),
                         ('AE', ae_reducers[2])]}
fig = plot_reconstructions(x_grid, y_test[idx], recons, titles=titles)
add_caption(fig, "Trois courbes de test et leurs reconstructions pour d = 2 : "
                 "PCA, Kernel PCA et autoencodeur.")
plt.show()

# 4) t-SNE : plongement 2-D des courbes du train (visualisation seulement)
t0 = time.perf_counter()
Z_tsne = TSNE(n_components=2, init='pca', random_state=42).fit_transform(y_train)
time_tsne = time.perf_counter() - t0
fig = plot_latent_space(Z_tsne, X_train, title="Plongement t-SNE (2-D) des courbes du train")
add_caption(fig, "Plongement t-SNE des courbes du train, coloré par μ puis par σ².")
plt.show()

# 5) Tableau comparatif des modèles
pca_d = dict(zip(d_values, mse_pca_test))
ae_d = dict(zip(d_values_ae, mse_ae_test))
kpca_d = dict(zip(d_values_ae, mse_kpca_test))
records = [
    {'méthode': 'PCA', 'type': 'linéaire',
     'EQM test d=2': pca_d[2], 'EQM test d=5': pca_d[5], 'EQM test d=10': pca_d[10],
     'temps entraînement d=2 (s)': pca_reducers[2].fit_time_,
     'encode de nouvelles courbes': 'oui', 'décodeur z → courbe': 'oui (exact)',
     'utilisable dans le pipeline': 'oui'},
    {'méthode': 'Kernel PCA (RBF)', 'type': 'non linéaire (noyau)',
     'EQM test d=2': kpca_d[2], 'EQM test d=5': kpca_d[5], 'EQM test d=10': kpca_d[10],
     'temps entraînement d=2 (s)': kpca_reducers[2].fit_time_,
     'encode de nouvelles courbes': 'oui', 'décodeur z → courbe': 'oui (appris, approché)',
     'utilisable dans le pipeline': 'oui'},
    {'méthode': 'Autoencodeur', 'type': 'non linéaire (réseau)',
     'EQM test d=2': ae_d[2], 'EQM test d=5': ae_d[5], 'EQM test d=10': ae_d[10],
     'temps entraînement d=2 (s)': ae_reducers[2].fit_time_,
     'encode de nouvelles courbes': 'oui', 'décodeur z → courbe': 'oui (réseau décodeur)',
     'utilisable dans le pipeline': 'oui'},
    {'méthode': 't-SNE', 'type': 'non linéaire (voisinages)',
     'EQM test d=2': None, 'EQM test d=5': None, 'EQM test d=10': None,
     'temps entraînement d=2 (s)': time_tsne,
     'encode de nouvelles courbes': 'non', 'décodeur z → courbe': 'non',
     'utilisable dans le pipeline': 'non'},
]
display(models_table(records))

**Q1.4.** *(Bonus)* Le Kernel PCA améliore-t-il la reconstruction ? Qu'en est-il du coût de calcul ? Pourquoi t-SNE/UMAP ne peuvent-ils pas être utilisés tels quels dans le pipeline (pensez à l'étape GPR → décodeur) ?

**Réponse Q1.4 :**

**Kernel PCA.** Elle améliore la reconstruction par rapport à la PCA aux petites dimensions : EQM test de $4{,}9 \cdot 10^{-4}$ contre $1{,}06 \cdot 10^{-3}$ à $d = 2$ ($2{,}2$ fois moins), et $1{,}5$ fois moins à $d = 5$. L'avantage disparaît à $d \geq 10$, et elle reste environ 3 fois moins précise que l'autoencodeur à $d = 2$.

Son ajustement est rapide (≈ **1,1 s**, comparable à la PCA et bien moins que les 64 à 70 s de l'AE), mais elle a un inconvénient majeur : **son coût croît fortement avec le nombre $n$ de courbes d'entraînement**. Il faut stocker **une matrice $n \times n$**, la diagonalisation coûte **$O(n^3)$**, et chaque encodage ou décodage nécessite de comparer le point aux $n$ courbes du train.

**Pourquoi on ne peut pas utiliser t-SNE / UMAP ?** Parce que le pipeline exige deux opérations : prédire un code $z$ à partir de $(\mu, \sigma^2)$, puis **décoder** $z$ en courbe. Or t-SNE ne calcule qu'un plongement des données d'entraînement (`fit_transform`) : il ne sait ni encoder une nouvelle courbe (pas de `transform`), ni surtout reconstruire une courbe à partir d'un point de l'espace latent (pas d'`inverse_transform`). Le $z$ prédit par la GPR ne pourrait donc jamais être transformé en courbe. De plus, t-SNE ne préserve que les voisinages locaux, pas les distances : ses coordonnées varient d'une exécution à l'autre et ne forment pas une fonction régulière de $(\mu, \sigma^2)$. UMAP propose un `transform` et un `inverse_transform`, mais ce dernier n'est qu'une approximation par interpolation entre points d'entraînement, sans garantie de régularité. Ces méthodes conviennent à la visualisation, pas au rôle de décodeur d'un méta-modèle.

## Partie 2 — Régression GPR dans l'espace latent

On apprend maintenant la régression **entrées → codes latents** :

$$\psi : (\mu, \sigma^2) \in \mathbb{R}^2 \;\longrightarrow\; \mathbf{z} \in \mathbb{R}^d,$$

par un **processus gaussien multi-sorties** (une GPR indépendante par dimension latente, ou une seule GPR sur les $d$ sorties). Complétez le squelette `src/gpr_model.py`.

Consignes :
1. **Noyaux à comparer** : `RBF`, `Matern(2.5)`, `RationalQuadratic`, chacun multiplié par `ConstantKernel` et additionné d'un `WhiteKernel` (bruit). 
2. **Optimisation des hyperparamètres** : laisser `GaussianProcessRegressor` maximiser la log-vraisemblance marginale, avec `n_restarts_optimizer` redémarrages pour éviter les maxima locaux ; justifier le nombre de redémarrages choisi.
3. Pour **chaque réducteur** retenu en Partie 1 (au minimum AE avec votre meilleur $d$) et **chaque noyau** : entraînez la GPR sur le train, puis
   - mesurez l'erreur dans l'espace latent : $\mathrm{RMSE}_z = \sqrt{\mathrm{MSE}(\hat{\mathbf z}, \mathbf z)}$ sur le test ;
   - **décodez** les prédictions $\hat{\mathbf z}$ et mesurez l'EQM de courbe correspondante sur le test.

```python
# --- snippet d'aide : grille de noyaux ---
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel as C, WhiteKernel

kernels = {
    "RBF":   C(1.0) * RBF(1.0)            + WhiteKernel(1e-3),
    "Matern": C(1.0) * Matern(nu=2.5)     + WhiteKernel(1e-3),
    "RQ":    C(1.0) * RationalQuadratic() + WhiteKernel(1e-3),
}
gpr = GaussianProcessRegressor(kernel=kernels["RBF"], alpha=1e-5,
                               n_restarts_optimizer=3, normalize_y=True)
gpr.fit(X_train, Z_train)          # Z_train : codes latents du train, shape (n_train, d)
Z_pred, Z_std = gpr.predict(X_test, return_std=True)
```

In [ ]:
from src.gpr_model import KERNELS, evaluate_gpr, paired_comparison
from src.visualization import results_table, plot_reconstructions, add_caption

N_RESTARTS = 0               # justifié en cellule ci dessous
ae2 = ae_reducers[2]         

gpr_models, gpr_records, gpr_errors = {}, [], {}
for k in KERNELS:
    gpr, rec, err = evaluate_gpr(ae2, X_train, y_train, X_test, y_test,
                                 kernel_name=k, n_restarts=N_RESTARTS)
    gpr_models[k], gpr_errors[k] = gpr, err
    gpr_records.append({'noyau': k, **rec})
display(results_table(gpr_records, index='noyau'))

# Significativité : chaque noyau comparé au RBF sur les mêmes 400 courbes de test
sig = []
for k in ['Matern', 'RQ']:
    pc = paired_comparison(gpr_errors[k], gpr_errors['RBF'])
    sig.append({'comparaison': f'{k} / RBF', 'ratio des EQM': pc['ratio'],
                'ratio IC95 bas': pc['ratio_ic95'][0], 'ratio IC95 haut': pc['ratio_ic95'][1],
                'part des courbes où le noyau gagne': pc['part_a_meilleur'],
                'p-valeur (Wilcoxon)': pc['p_wilcoxon']})
display(results_table(sig, index='comparaison'))

v, e = X_test[:, 1], gpr_errors['RBF']
print(f"{e[v <= np.quantile(v, 0.1)].sum() / e.sum():.0%} de l'EQM vient des 10 % de courbes les plus "
      f"étroites, {e[np.argmin(v)] / e.sum():.0%} de la seule courbe sigma2 = {v.min():.3f}")

recons = {f'AE d=2 + GPR {k}': ae2.decoder(gpr_models[k].predict(X_test[idx])) for k in KERNELS}
fig = plot_reconstructions(x_grid, y_test[idx], recons, titles=titles)
add_caption(fig, "Courbes de test et prédictions du méta-modèle (GPR puis décodeur de l'AE, d = 2) "
                 "pour les trois noyaux, à partir des seuls paramètres (μ, σ²).")
plt.show()

In [ ]:
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
k_sans_white = C(1.0) * RBF([1.0, 1.0], length_scale_bounds=(1e-2, 1e2))
ref = next(r for r in gpr_records if r['noyau'] == 'RBF')
q22 = [{'configuration': 'WhiteKernel + alpha = 1e-5 (référence)',
        **{k: v for k, v in ref.items() if k != 'noyau'}, 'résultat': 'ok'}]
for label, kw in [('sans WhiteKernel, alpha = 1e-10', dict(kernel=k_sans_white, alpha=1e-10)),
                  ('sans WhiteKernel, alpha = 0', dict(kernel=k_sans_white, alpha=0.0)),
                  ('WhiteKernel + alpha = 1e-1 (trop grand)', dict(kernel_name='RBF', alpha=1e-1))]:
    try:
        _, rec, _ = evaluate_gpr(ae2, X_train, y_train, X_test, y_test, n_restarts=N_RESTARTS, **kw)
        q22.append({'configuration': label, **rec, 'résultat': 'ok'})
    except Exception as e:
        q22.append({'configuration': label, 'résultat': f'échec : {type(e).__name__} (matrice non définie positive)'})
display(results_table(q22, index='configuration'))
print('Bruit estimé par le WhiteKernel :', gpr_models['RBF'].kernel_.k2)

In [ ]:
from sklearn.gaussian_process.kernels import WhiteKernel
noyau_sujet = C(1.0) * RBF(1.0) + WhiteKernel(1e-3)        # noyau d'origine du sujet
essais = []
for label, kw, nr in [('(mu, log sigma2), RBF borné', dict(kernel_name='RBF'), 0),
                      ('(mu, log sigma2), RBF borné', dict(kernel_name='RBF'), 3),
                      ('entrée brute (mu, sigma2), noyau du sujet', dict(kernel=noyau_sujet, log_variance=False), 0)]:
    gpr, rec, _ = evaluate_gpr(ae2, X_train, y_train, X_test, y_test, n_restarts=nr, **kw)
    essais.append({'configuration': label, 'n_restarts': nr, 'EQM décodée': rec['EQM décodée'],
                   'temps GPR (s)': rec['temps GPR (s)'],
                   'log-vraisemblance': gpr.log_marginal_likelihood_value_,
                   'noyau optimisé': str(gpr.kernel_)})
display(results_table(essais, index=['configuration', 'n_restarts']))

In [ ]:
from src.visualization import plot_decoded_vs_d
sweep = []
for d in [2, 3, 5, 10, 20]:
    _, rec, _ = evaluate_gpr(ae_reducers[d], X_train, y_train, X_test, y_test,
                             kernel_name='RBF', n_restarts=N_RESTARTS)
    sweep.append({'d': d, **rec})
display(results_table(sweep, index='d'))
fig = plot_decoded_vs_d({'AE': {col: [r[col] for r in sweep]
                                for col in ['d', 'EQM décodée', 'EQM reconstruction', 'RMSE latente relative']}})
add_caption(fig, "Autoencodeur + GPR (noyau RBF) : EQM de courbe après GPR et décodage, EQM de "
                 "reconstruction seule, et RMSE latente relative, en fonction de d.")
plt.show()

# Méta-modèle retenu pour la Partie 3 : AE d = 2 + le noyau à l'EQM décodée la plus faible
best_kernel = min(gpr_records, key=lambda r: r['EQM décodée'])['noyau']
best_gpr, best_decoder = gpr_models[best_kernel], ae2.decoder
print('Méta-modèle retenu : AE d = 2 + GPR', best_kernel)

**Q2.1.** Quel noyau donne les meilleures prédictions latentes ? L'écart entre noyaux est-il significatif ?

**Q2.2.** Quel est le rôle du `WhiteKernel` et de `alpha` ? Que se passe-t-il si on les supprime ? (Testez sur un cas.)

**Q2.3.** Comment l'erreur décodée dépend-elle de la dimension latente $d$ ? Y a-t-il un « sweet spot » ?

**Réponse Q2.1 :**
Le noyau **RBF** donne les meilleures prédictions : RMSE latente de $1{,}98$ (soit $0{,}30$ fois l'écart-type des codes) et EQM décodée de $2{,}44 \cdot 10^{-4}$, contre $2{,}56 \cdot 10^{-4}$ pour Matern et $2{,}89 \cdot 10^{-4}$ pour RQ.

L'écart n'est cependant **pas significatif**. Comparés courbe par courbe sur les 400 courbes de test, Matern et RQ ont des EQM $1{,}05$ et $1{,}18$ fois plus élevées que RBF, mais les intervalles de confiance à 95 % de ces rapports contiennent 1 ($[0{,}45 ; 1{,}12]$ et $[0{,}57 ; 1{,}85]$). Chaque noyau fait mieux que RBF sur environ une courbe sur deux (53 % et 50 %), et le test de Wilcoxon ne détecte aucune différence ($p = 0{,}75$ et $0{,}88$).

Ces intervalles sont larges parce que l'erreur se concentre sur très peu de courbes : les 10 % de courbes les plus étroites portent la quasi-totalité de l'EQM, et la seule courbe $\sigma^2 = 0{,}029$ (plus étroite que toutes celles du train, donc extrapolée) en représente 54 %. Le choix du noyau compte donc peu. On retient le RBF, le plus précis, avec un temps d'ajustement comparable à Matern (54 s contre 48 s) et bien inférieur à RQ (129 s).

**Réponse Q2.2 :** *(à compléter)*

**Réponse Q2.3 :** *(à compléter)*

## Partie 3 — Probabilité de dépassement d'un seuil

On assemble le méta-modèle complet et on estime, par Monte Carlo, la probabilité que la courbe aléatoire dépasse le seuil $\tau = 0.1$ en chaque position $x$ :

$$\hat p(x) = \frac{1}{L}\sum_{l=1}^{L} \mathbf{1}\big\{\hat y^{(l)}(x) \geq \tau\big\}, \qquad \hat{\mathbf y}^{(l)} = f_\theta\big(\psi(\mu^{(l)}, (\sigma^2)^{(l)})\big),$$

où $(\mu^{(l)}, (\sigma^2)^{(l)})$ sont $L = 20\,000$ tirages de la loi d'entrée. Complétez `mc_exceedance` dans `src/exceedance.py`.

### Référence exacte (fournie)

Pour cette loi d'entrée (moyenne uniforme, variance gamma), la probabilité exacte s'obtient par intégration numérique à une dimension : conditionnellement à $\sigma^2 = v$, l'événement $\{\phi(x;\mu,\sigma^2)\geq\tau\}$ équivaut à $\mu \in [x-c(v),\, x+c(v)]$ avec $c(v)=\sqrt{2v\ln\frac{1}{\tau\sqrt{2\pi v}}}$ (nécessite $v < v_\tau = \frac{1}{2\pi\tau^2}$), d'où

$$E(x) = \int_0^{v_\tau} \frac{\big|[x-c(v),\,x+c(v)]\big|}{8}\; \gamma(v; 2, 2)\, dv .$$

La cellule ci-dessous calcule cette référence : **exécutez-la, puis répondez aux questions après avoir implémenté la partie Monte Carlo.**

In [ ]:
# --- reference exacte (FOURNIE, ne pas modifier) ---
from scipy.integrate import quad
from scipy.stats import gamma

tau = 0.1
v_tau = 1.0 / (2.0 * np.pi * tau**2)

def exact_exceedance(x):
    def integrand(v):
        c = np.sqrt(2.0 * v * np.log(1.0 / (tau * np.sqrt(2.0 * np.pi * v))))
        lo, hi = max(x - c, mu_min), min(x + c, mu_max)
        return max(hi - lo, 0.0) / (mu_max - mu_min) * gamma.pdf(v, a=k_shape, scale=theta_scale)
    return quad(integrand, 0.0, v_tau, epsabs=1e-12)[0]

p_exact = np.array([exact_exceedance(x) for x in x_grid])
print(f"v_tau = {v_tau:.4f} ; E(x) in [{p_exact.min():.4f}, {p_exact.max():.4f}]")

plt.figure(figsize=(10, 4))
plt.plot(x_grid, p_exact, 'k-', lw=2, label=r'reference exacte $E(x)$')
plt.xlabel(r'$x$'); plt.ylabel(r'$\mathbb{P}(\phi(x)\geq\tau)$')
plt.title(f'Probabilite de depassement exacte, tau = {tau}')
plt.legend(); plt.grid(alpha=0.3); plt.show()

Consignes :
1. Complétez `mc_exceedance(gpr, decoder, n_samples, tau)` dans `src/exceedance.py` : tirage des paramètres selon la loi d'entrée, prédiction GPR, décodage, moyenne des indicateurs.
2. Tracez sur un même graphe : la référence exacte, $\hat p(x)$ pour votre meilleur méta-modèle de la Partie 2, et — à titre de contrôle — la probabilité empirique obtenue par Monte Carlo sur le **vrai simulateur** (courbes exactes, mêmes tirages).
3. Quantifiez : erreur en aire $\int |\hat p - E|\,dx$, erreur max, corrélation entre les profils.

```python
# --- snippet d'aide ---
mu_l    = np.random.uniform(mu_min, mu_max, L)
var_l   = np.random.gamma(k_shape, theta_scale, L)
X_mc    = np.column_stack([mu_l, var_l])
Z_mc    = gpr.predict(X_mc)                # prediction latente
curves  = decoder.predict(Z_mc)            # ou decoder(Z_mc) selon votre librairie
p_hat   = (curves >= tau).mean(axis=0)
```

In [ ]:
# TODO 3 : estimation de p_hat par le meta-modele, comparaison a la reference exacte
# et au MC sur le vrai simulateur. Metriques d'erreur.
# VOTRE CODE ICI

**Q3.1.** Le profil $\hat p(x)$ du méta-modèle est-il conforme à la référence exacte ? Repérez les zones de plus grande erreur et expliquez-les (pensez aux bords du domaine des paramètres et à la reconstruction).

**Q3.2.** Faites varier $L \in \{10^3, 10^4, 2\cdot 10^4\}$ : comment évolue l'erreur ? Séparez, en vous aidant du contrôle « vrai simulateur », la part d'erreur due au bruit de Monte Carlo de celle due au méta-modèle.

**Q3.3.** *(Bonus)* Reprenez la question avec $\tau \in \{0.05, 0.2\}$. Le méta-modèle reste-t-il précis pour les seuils extrêmes ?

**Réponse Q3.1 :** *(à compléter)*

**Réponse Q3.2 :** *(à compléter)*

**Réponse Q3.3 :** *(à compléter)*

## Partie 4 — Synthèse et discussion

Consignes :
1. Construisez un **tableau récapitulatif** : méthode de réduction × dimension $d$ × noyau → EQM reconstruction, RMSE latente, EQM décodée, erreur en aire sur $E(x,\tau)$.
2. Choisissez **un** méta-modèle final et justifiez ce choix.
3. Discutez ce qui changerait si le simulateur était réellement coûteux (quelques minutes par évaluation) : quel gain de temps le méta-modèle apporte-t-il pour produire $L = 20\,000$ courbes ?

In [ ]:
# TODO 4 : tableau synthetique et figure de synthese (profils p_hat pour les meilleurs modeles).
# VOTRE CODE ICI

**Q4.1.** Quel méta-modèle recommandez-vous ? Argumentez avec au moins trois critères quantifiés.

**Réponse Q4.1 :** *(à compléter)*

---

## Avant de rendre — checklist

- [ ] Toutes les cellules s'exécutent de haut en bas sans erreur (*Kernel → Restart & Run All*).
- [ ] Les fonctions/ classes de fond sont dans `src/*.py` (pas de grosses fonctions dans le notebook).
- [ ] Toutes les questions `Q…` sont traitées dans les cellules « Réponse ».
- [ ] Figures titrées/légendées, graines fixées, choix PyTorch/tensorflow annoncé.
- [ ] Dépôt effectué sur Moodle (notebook + `src/`), **aucun rapport à joindre**.
